# **Diplomado IA: Inteligencia Artificial II - Parte 1**. <br> Laboratorio 14: Transformers
---
---

**Profesor:**
- Felipe del Río

**Ayudante:**
- Bianca del Solar
---
---

# **Instrucciones Generales**

El siguiente práctico será **individual**. Solo uno debe realizar la entrega. El formato de entregar es el **archivo .ipynb con todas las celdas ejecutadas**. Todas las preguntas deben ser respondida en celdas de texto. No se aceptará el _output_ de una celda de código como respuesta.

**Nombre:** Vicente Zapata

El siguiente práctico cuenta con 3 secciones donde cada una contendrá 1 o más actividades a realizar. Algunas actividades correspondrán a escribir código y otras a responder preguntas.

**Importante.** Para facilitar su ejecución, cada sección puede ser ejecutada independientemente.

Se recomienda **fuertemente** revisar las secciones donde se entrega código porque algunas actividades de código pueden reutilizar el mismo código pero con cambios en algunas líneas.



# **Agenda**

>[Diplomado IA: Inteligencia Artificial II - Parte 1.  Laboratorio 14: Transformers](#scrollTo=tHopPtVaNF1K)

>[Instrucciones Generales](#scrollTo=uIdAKAdELPSl)

>[Agenda](#scrollTo=kEloa5uXLIPK)

>[Parte I: Inspeccionando las atenciones de un Transformer](#scrollTo=ZS3cYFT2TWB9)

>>[Preámbulo](#scrollTo=0dBeU4b818s4)

>>[Representación del Input para BERT](#scrollTo=WlA3k2R7gPWV)

>>[NER](#scrollTo=ukXvd54RjC_4)

>>[Visualización de Atenciones](#scrollTo=M7aoVKrWlLbp)

>>>[Head View](#scrollTo=Z4COgQK0BAsp)

>>>[Model View](#scrollTo=J9e2nT-FD_kH)

>>>[Neuron View](#scrollTo=in3biNv0pJV_)

>>>[Actividad 1](#scrollTo=7krb_CeKqANH)

>>[Actividad 2: Atenciones en el Decoder](#scrollTo=flpPAFaJfm5H)

>>[Traducción](#scrollTo=oyk-K3pjXrwX)



# Parte I: Inspeccionando las atenciones de un Transformer



En este laboratorio exploraremos un modelo de Transformer, BETO, el cual está basado en BERT y preentrenado en un corpus en español. Utilizaremos un modelo que ya fue refinado para la tarea de NER en español.

**NER (*Named Entity Recognition*)**, es una tarea que consiste en encontrar y detectar entidades nombradas (personas, organizaciones, etc.) dentro de un texto, además de clasificar a que tipo corresponde. Es sumemente útil para identificar que entidades están presentes en nuestros textos y poder un análisis más profundo en base a ellos.


## Preámbulo

Primero debemos descargar, instalar e importar las distintas librerías que utilizaremos para este laboratorio.

In [1]:
!pip install -q bertviz==1.4.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 157.6/157.6 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.6/140.6 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.6/14.6 MB 90.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 4.3 MB/s eta 0:00:00


In [2]:
import string

import IPython
from bertviz import head_view, model_view
from bertviz.neuron_view import show
import spacy
from spacy.vocab import Vocab
from spacy.tokens import Doc, Span
import torch
from transformers import AutoTokenizer, AutoModelForTokenClassification
from transformers import BertTokenizer, BertModel

También crearemos algunas funciones auxiliares para el laboratorio.

In [3]:
# Funciones auxiliares para la visualización de los resultados del modelo para NER

LABEL_LIST = [
    "B-LOC",    # Beginning of a location right after another location
    "B-MISC",   # Beginning of a miscellaneous entity right after another miscellaneous entity
    "B-ORG",    # Beginning of an organisation right after another organisation
    "B-PER",    # Beginning of a person's name right after another person's name
    "I-LOC",    # Location
    "I-MISC",   # Miscellaneous entity
    "I-ORG",    # Organisation
    "I-PER",    # Person's name
    "O"         # Outside of a named entity
]

# Construimos el Doc para uso de Spacy
def build_doc(tokens, spaces, vocab):
    tokens = [token.replace('##', '') for token in tokens] # Remove ## start in word pieces
    vocab = Vocab(strings=vocab)
    doc = Doc(vocab=vocab, words=tokens, spaces=spaces)
    return doc

# Obtenemos la lista de `spaces`, con `True` para los tokens que están seguidos de espacios
def get_spaces(tokens):
    spaces = []
    for i, token in enumerate(tokens):
        if i + 1 == len(tokens):
            spaces.append(False)
            break

        next_token = tokens[i+1]
        if next_token in string.punctuation:
            spaces.append(False)
            continue
        if next_token.startswith('##'):
            spaces.append(False)
            continue

        spaces.append(True)

    return spaces

# Transformar de una lista de entidades a un diccionarion con las entidades
# interesantes y sus ubicaciones
def get_entity_locs(entities):
    def gettype(ent):
        return ent.replace('B-', '').replace('I-', '')

    ents = []
    start, end = 0, 0
    type_ = ''
    for i, ent in enumerate(entities):
        if ent == 'O':
            if type_:
                ents.append((start, end, type_))
            start, end = 0, 0
            type_ = ''

        if ent.startswith('B-'):
            if type_:
                ents.append((start, end, type_))
            start, end = i, i
            type_ = gettype(ent)

        if type_ and ent.startswith('I-') and gettype(ent) == type_:
            end = i

        if i + 1 == len(entities) and type_:
            ents.append((start, end, type_))

    return ents

# Añadimos las entidades al objeto Doc
def add_entities(doc, entity_locs):
    doc_ents = []
    for start, end, type_ in entity_locs:
        span = Span(doc, start=start, end=end+1, label=type_)
        doc_ents.append(span)

    doc.ents = doc_ents
    return doc

# Integramos el pipeline completo para visualizar las entidades
def build_for_vis(tokens, predictions, special_tokens=None):
    special_tokens = [] if special_tokens is None else special_tokens

    ents = [LABEL_LIST[prediction] for token, prediction in zip(tokens, predictions) if token not in special_tokens]
    tokens = [token for token in tokens if token not in special_tokens]

    spaces = get_spaces(tokens)
    vocab = list(tokenizer.vocab.keys())
    doc = build_doc(tokens, spaces, vocab=vocab)

    entity_locs = get_entity_locs(ents)
    doc = add_entities(doc, entity_locs)
    return doc

# Funciones auxiliares para la visualización de atenciones

def call_html(view='head'):
    assert view in ['head', 'model', 'neuron']
    d3_version = {
        'head': '3.5.8', 'model': '5.7.0', 'neuron': '5.7.0'
    }[view]
    load_libs_html = f'''
            <script src="/static/components/requirejs/require.js"></script>
            <script>
            requirejs.config({{
                paths: {{
                base: '/static/base',
                "d3": "https://cdnjs.cloudflare.com/ajax/libs/d3/{d3_version}/d3.min",
                jquery: '//ajax.googleapis.com/ajax/libs/jquery/2.0.0/jquery.min',
                }},
            }});
            </script>
            '''
    display(IPython.core.display.HTML(load_libs_html))

## Representación del Input para BERT

Primero que todo, creamos el modelo y tokenizador a utilizar. Como vimos en clases, un modelo del tipo BERT requiere formatear la entrada agregando tokens especiales que cumplen algunas funciones. Para lograr este formato de forma sencilla, utilizaremos el tokenizador correspondiente.

In [4]:
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')

model_type = 'bert'
model_version = 'mrm8488/bert-spanish-cased-finetuned-ner' # Pesos a utilizar

tokenizer = AutoTokenizer.from_pretrained(model_version)
model = AutoModelForTokenClassification.from_pretrained(model_version, output_attentions=True)
model.to(device)

special_tokens = tokenizer.special_tokens_map.values()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/136 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/829 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/439M [00:00<?, ?B/s]

Some weights of the model checkpoint at mrm8488/bert-spanish-cased-finetuned-ner were not used when initializing BertForTokenClassification: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForTokenClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


model.safetensors:   0%|          | 0.00/439M [00:00<?, ?B/s]

In [5]:
text = 'Eduardo Vargas le metió un gol a España en el mundial de Brasil.'

tokens = tokenizer.tokenize(tokenizer.decode(tokenizer.encode(text)))
inputs = tokenizer(text, return_tensors="pt")
inputs.to(device)
None

A continuación utilizaremos el tokenizador que nos provee la librería Transformers para entender mejor como se construye el input para un modelo basado en BERT.

Podemos observar cual es el resultado de agregar estos tokens a nuestro texto. Para esto usamos el *hack*, de codificar y luego decodificar nuestro texto.

In [6]:
tokenizer.decode(tokenizer.encode(text))

'[CLS] Eduardo Vargas le metió un gol a España en el mundial de Brasil. [SEP]'

Sabemos que BERT está pensado para utilizar varias frases en el input, como por ejemplo para tareas de pregunta-respuesta podemos ver el resultado de la misma forma, entregandole el segundo texto como input al tokenizador.

In [7]:
text2 = 'Y Alexis Sánchez a Brasil en octavos, en Belo Horizonte'
tokenizer.decode(tokenizer.encode(text, text2))

'[CLS] Eduardo Vargas le metió un gol a España en el mundial de Brasil. [SEP] Y Alexis Sánchez a Brasil en octavos, en Belo Horizonte [SEP]'

También, podemos ver la forma en que se hace tokenización a nivel de sub-palabras, es decir que existen tokens que corresponden a partes de palabras en vez de la palabra completa.

In [8]:
tokens

['[CLS]',
 'Eduardo',
 'Vargas',
 'le',
 'metió',
 'un',
 'gol',
 'a',
 'España',
 'en',
 'el',
 'mundial',
 'de',
 'Brasil',
 '.',
 '[SEP]']

El tokenizador nos ayuda también a preparar los otros inputs que entrarán al modelo, como lo son por ejemplo los *segment embeddings*. A continuación se muestran tanto para una frase como para dos.



![](https://miro.medium.com/max/1000/1*EKzyGf_l0e57XN491_YAyg.png)

In [9]:
sample_input = tokenizer(text, return_tensors="pt")
sample_input['token_type_ids']

tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]])

In [10]:
sample_input2 = tokenizer(text, text2, return_tensors="pt")
sample_input2['token_type_ids']

tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1]])

## NER

Ahora utilicemos el modelo que creamos para la tarea de NER y veamos las predicciones que realiza

In [11]:
outputs = model(**inputs)[0]
predictions = torch.argmax(outputs, dim=2)
predictions = predictions[0].tolist()

In [12]:
labeled_text = build_for_vis(tokens, predictions, special_tokens=special_tokens)

In [13]:
from spacy import displacy

displacy.render(labeled_text, style="ent", jupyter=True)

Puede utilizar el código a continuación para obtener una visualización en base al texto que ud. desee. Para esto debe asignarlo a la variable `texto`.

In [14]:
texto = "A Pedro le gusta comer en McDonald's de USA" #@param {type:"string"}

if texto:
    tokens = tokenizer.tokenize(
        tokenizer.decode(tokenizer.encode(texto)))
    inputs = tokenizer(texto, return_tensors="pt")
    inputs.to(device)

    outputs = model(**inputs)[0]
    predictions = torch.argmax(outputs, dim=2)
    predictions = predictions[0].tolist()

    labeled_text = build_for_vis(
        tokens, predictions, special_tokens=special_tokens)
    displacy.render(labeled_text, style="ent", jupyter=True)

## Visualización de Atenciones

Como sabemos, la arquitectura de un modelo Transformer está construída en gran parte sobre atenciones. Esto nos da la posibilidad de poder visualizar estas atenciones para poder hacernos una idea de lo que está pasando dentro del modelo mismo. Para generar las visualizaciones nos ayudará el repositiorio [BertViz](https://github.com/jessevig/bertviz).

**Importante.** Es posible que las visualizaciones a continuación no se carguen correctamente la primera vez que se ejecuta la celda, en dicho caso intenten ejecutar nuevamente la celda.

### Head View

Primero usamos la visualización del tipo `head_view` para ver los patrones de atención de una o más *heads* para una capa dada.

En esta visualización podemos ver la representación de la palabra que está siendo actualizada (*query*) a la **izquierda**, mientras que las palabras a las cuales se le pone atención (*key, value*) a la **derecha** de la visualización.

Con el *dropdown* podemos cambiar la capa en la cual nos fijamos y cada color significa una *head* distinta del modelo.

In [15]:
# Head View
texto = 'Eduardo Vargas le metió un gol a España en el mundial de Brasil.'

tokens = tokenizer.tokenize(tokenizer.decode(tokenizer.encode(texto)))
inputs = tokenizer(texto, return_tensors="pt")
inputs.to(device)

token_type_ids = inputs['token_type_ids']
input_ids = inputs['input_ids']

attention = model(**inputs)[-1]
input_id_list = input_ids[0].tolist() # Batch index 0
tokens = tokenizer.convert_ids_to_tokens(input_id_list)
call_html(view='head')

head_view(attention, tokens)

<IPython.core.display.Javascript object>

Puede utilizar el código a continuación para obtener una visualización en base al texto que ud. desee. Para esto debe asignarlo a la variable `texto`.

In [16]:
# Head View
texto = '' #@param {type:"string"}

if texto:
    tokens = tokenizer.tokenize(
        tokenizer.decode(tokenizer.encode(texto)))
    inputs = tokenizer(texto, return_tensors="pt")
    inputs.to(device)

    token_type_ids = inputs['token_type_ids']
    input_ids = inputs['input_ids']

    attention = model(**inputs)[-1]
    input_id_list = input_ids[0].tolist() # Batch index 0
    tokens = tokenizer.convert_ids_to_tokens(input_id_list)
    call_html(view='head')

    head_view(attention, tokens)

### Model View

A continuación  tenemos una visualización muy similar a  la anterior, sin embargo esta muestra cada capa y head del modelo por separado.

Cada fila está asociada a una capa del modelo (en orden creciente) mientras que cada columna corresponde a una *head* distinta en dicha capa.

In [17]:
texto = 'Eduardo Vargas le metió un gol a España en el mundial de Brasil.'

tokens = tokenizer.tokenize(tokenizer.decode(tokenizer.encode(texto)))
inputs = tokenizer(texto, return_tensors="pt")
inputs.to(device)

token_type_ids = inputs['token_type_ids']
input_ids = inputs['input_ids']
attention = model(**inputs)[-1]
input_id_list = input_ids[0].tolist() # Batch index 0
tokens = tokenizer.convert_ids_to_tokens(input_id_list)
call_html(view='model')

model_view(attention, tokens)

<IPython.core.display.Javascript object>

A continuación puede utilizar el código a continuación para obtener una visualización en base al texto que ud. desee. Para esto debe asignarlo a la variable `texto`.

In [18]:
texto = '' #@param {type:"string"}

if texto:
    tokens = tokenizer.tokenize(
        tokenizer.decode(tokenizer.encode(texto)))
    inputs = tokenizer(texto, return_tensors="pt")
    inputs.to(device)

    token_type_ids = inputs['token_type_ids']
    input_ids = inputs['input_ids']
    attention = model(**inputs)[-1]
    input_id_list = input_ids[0].tolist() # Batch index 0
    tokens = tokenizer.convert_ids_to_tokens(input_id_list)
    call_html(view='model')

    model_view(attention, tokens)

### Neuron View

A continuación tenemos una visualización muy similar a las anteriores, principalmente a la primera. Sin embargo, en esta podemos visualizar de mucho mejor forma las activaciones que están generando para cada una de las representaciones y como estas afectan la atención.

Este gráfico se interpreta de la misma manera que el primero (*Head View*), sin embargo podemos hacer click en las palabras de la izquierda para visualizar el cálculo de cada atención.

**Importante.** Este modelo trabaja con una versión modificada de la librería Transformer que hemos utilizada (notar los `import`'s a continuación), es por eso que las frases a utilizar son en inglés pues solo contamos con los modelos pre-entrenados base.

In [19]:
from bertviz.transformers_neuron_view import BertModel as VizBertModel, BertTokenizer as VizBertTokenizer
from bertviz.neuron_view import show

In [20]:
# Ojo: Puede haber problemas con la visualización si es que las oraciones son muy largas.
sentence_a = "Alexis scored against Brazil in the World Cup."
sentence_b = "Pinilla's shot struck the crossbar in that match."

In [21]:
nv_model_type = 'bert'
nv_model_version = 'bert-base-uncased'

do_lower_case = 'uncased' in nv_model_version
tokenizer = VizBertTokenizer.from_pretrained(nv_model_version, do_lower_case=do_lower_case)
model = VizBertModel.from_pretrained(nv_model_version)
call_html(view='neuron')
show(model, nv_model_type, tokenizer, sentence_a, sentence_b)

Output hidden; open in https://colab.research.google.com to view.

### Actividad 1

Descomente el código a continuación y elija una versión de BERT para probar la misma visualización que la obtenida anteriormente. Para esto debe definir correctamente la variable `nv_model_version`.

Basta con la ejecución para la actividad. Debe ser una versión distinta a la ya utilizada, `'bert-base-uncased'`.

**Importante:** Por limitaciones del sistema, no elijan modelos muy pesados para la visualización pues probablemente se caerán, principalmente eviten los modelos con `large` en el nombre.

In [22]:
# nv_model_type = 'bert'
# nv_model_version = 'bert-base-uncased'

# do_lower_case = 'uncased' in nv_model_version
# tokenizer = VizBertTokenizer.from_pretrained(nv_model_version, do_lower_case=do_lower_case)
# model = VizBertModel.from_pretrained(nv_model_version)
# call_html(view='neuron')
# show(model, nv_model_type, tokenizer, sentence_a, sentence_b)

También responda **brevemente** las siguientes preguntas en base a la actividad realizada.

**1. En las visualizaciones de las atenciones que vimos más arriba. ¿Por qué siempre se muestra la misma oración, tanto la que atiende con la que es atendida?**

In [23]:
R = "En las visualizaciones de atenciones vemos la misma oración repetida porque estamos trabajando  con el ENCODER de BERT, que es un modelo bidireccional.  El encoder procesa TODA la secuencia de entrada simultáneamente y calcula las relaciones de  atención entre cada token y todos los demás tokens de la MISMA secuencia. Por eso:  - La fila izquierda (queries): representa los tokens que \"atienden\" o buscan información - La fila derecha (keys/values): representa los tokens a los que se les presta atención  Como es self-attention (auto-atención), cada token de la oración puede atender a cualquier otro  token de la MISMA oración, incluyéndose a sí mismo. Esto permite capturar relaciones contextuales  bidireccionales, que es la característica principal de BERT.  Si estuviéramos visualizando un modelo sequence-to-sequence completo (como un traductor con  encoder y decoder), veríamos dos oraciones diferentes en las cross-attention del decoder." #@param {type:"string"}

**2. En la visualización Neuron View ¿a qué corresponden los parámetros "Layer" y "Head"?**

Parámetro Layer:

In [24]:
R = "El parámetro \"Layer\" indica la CAPA del transformer que estamos visualizando.  Un transformer tiene múltiples capas apiladas (en BERT-base son 12 capas). Cada capa: - Procesa las representaciones de la capa anterior - Aplica mecanismos de atención y transformaciones feed-forward - Genera representaciones más abstractas y contextualizadas  Las capas iniciales (0-3) típicamente capturan: - Patrones sintácticos básicos - Relaciones gramaticales simples - Información de nivel superficial  Las capas intermedias (4-8) capturan: - Relaciones semánticas - Dependencias a mayor distancia  Las capas finales (9-11) capturan: - Información semántica de alto nivel - Representaciones abstractas para la tarea específica  Seleccionar diferentes valores de Layer nos permite ver cómo evoluciona la atención a través  de la profundidad del modelo." #@param {type:"string"}

Parámetro Head:

In [25]:
R = "El parámetro \"Head\" indica cuál de las CABEZAS DE ATENCIÓN (attention heads) estamos visualizando.  Cada capa de un transformer usa Multi-Head Attention, que significa que la atención se calcula  múltiples veces en paralelo con diferentes proyecciones. En BERT-base hay 12 heads por capa.  ¿Por qué múltiples heads? - Cada head puede aprender a enfocarse en DIFERENTES tipos de relaciones - Un head podría capturar relaciones sintácticas (sujeto-verbo) - Otro head podría capturar relaciones semánticas (co-referencias) - Otro podría enfocarse en relaciones de dependencia específicas  Beneficios: - DIVERSIDAD: Cada head aprende patrones distintos - ESPECIALIZACIÓN: Diferentes heads se especializan en diferentes aspectos lingüísticos - ROBUSTEZ: Si un head no captura algo importante, otros pueden compensarlo  Al cambiar el parámetro Head, podemos explorar qué tipo de relaciones cada head ha aprendido  a identificar durante el entrenamiento." #@param {type:"string"}

## Actividad 2: Atenciones en el Decoder

En esta actividad inspeccionaremos la atención cruzada que realiza el decoder sobre el encoder. Esta atención nos dará una mejor idea de que está usando el modelo para generar texto en el decoder.

Responda las preguntas a continuación.

## Traducción

Utilizaremos el ejemplo de atenciones en traducciones.

A continuación se muestra el diagrama de las visualizaciones de atención del modelo, enfocándonos en este caso en las atenciones que el decoder del transformer asigna sobre las salidas del encoder. En la literatura, este mecanismo suele denominarse **cross-attention**.


<img width="400" height="400" src="data:image/png;base64, ..." />

En el archivo de configuración de la red que produjo estas atenciones tenemos la siguiente información:

```
...
enc_layers: 6
dec_layers: 6
heads: 8
...
```
Al generar el gráfico anterior, solo lo estamos visualizando las cross attentions para una capa y una "head" del modelo. **¿Cuántos gráficos tenedremos si es que pudiesemos visualizar todas las cross-attentions?**

In [26]:
R = 48 # @param{type: "number"}

El decoder del Transformer, no solo utiliza este tipo de atenciones, también utiliza las del tipo self-attention. Si estuvieramos haciendo decoding de una traducción en el paso T=5, es decir generando el 5to token de salida , tal y como se muestra en la siguiente imagen.

![](https://www.guru99.com/images/1/111318_0848_seq2seqSequ1.png)

Si visualizaramos las atenciones de self-attention del decoder **¿Qué dimensiones tendría la matriz mostrada?**

In [30]:
R = '5x5' # @param {type: "string"}

**Responda si la siguiente afirmación es verdadera o falsa.**


En el decoder de un transformer, como lo vimos en clase, no es necesario enmascarar "el futuro".

In [31]:
R = 'Falso' #@param ["", "Verdadero", "Falso"]

En un Transformer, el positional encoding se utiliza como **única** fuente de información del orden de la secuencia.

In [32]:
R = 'Verdadero' #@param ["", "Verdadero", "Falso"]